# Train AURUM genre-v1 on a free Colab GPU

Runtime → **Change runtime type → T4 GPU**, then Run all.

Start with `small` (fast, ~6 roots) to validate end-to-end, then rerun with `medium` for the real 11-root `genre-v1`. Everything writes to `release/`, which the last cell zips for download.

In [ ]:
!nvidia-smi -L   # confirm a GPU is attached

In [ ]:
# Clone the repo. aurum-genre is PRIVATE: set a GitHub token (Colab: 🔑 Secrets → GH_TOKEN,
# a fine-grained PAT with Contents:read on tablackmore/aurum-genre). If the repo is public, drop the token.
import os
from google.colab import userdata
try:
    tok = userdata.get('GH_TOKEN')
    url = f'https://{tok}@github.com/tablackmore/aurum-genre.git'
except Exception:
    url = 'https://github.com/tablackmore/aurum-genre.git'
!git clone -q $url
%cd aurum-genre
!pip -q install -e '.[dev]'

In [ ]:
SUBSET = 'small'   # 'small' = fast validation; 'medium' = full genre-v1
EPOCHS = 30        # small: 30 · medium: 60
!bash scripts/download_fma.sh $SUBSET

In [ ]:
# Full pipeline: manifests → train (GPU) → eval (macro-AUC) → export genre.onnx → package release/
!python scripts/run_pipeline.py --subset $SUBSET --epochs $EPOCHS

In [ ]:
# Zip the release assets and download them.
!cd release && zip -q -r ../genre-v1.zip genre.onnx taxonomy.json thresholds.json mel_recipe.txt mel_golden.npz NOTICE license_manifest.csv
from google.colab import files
files.download('genre-v1.zip')